In [1]:
import os, zipfile, urllib.request
import pandas as pd

pd.set_option("display.max_columns", None)

BASE = "/content/EduPredict/data/raw"
os.makedirs(BASE, exist_ok=True)

urls = {
    "dropout": "https://archive.ics.uci.edu/static/public/697/predict+students+dropout+and+academic+success.zip",
    "performance": "https://archive.ics.uci.edu/static/public/320/student+performance.zip",
}

for name, url in urls.items():
    zip_path = os.path.join(BASE, f"{name}.zip")
    extract_dir = os.path.join(BASE, name)
    try:
        urllib.request.urlretrieve(url, zip_path)
        with zipfile.ZipFile(zip_path) as z:
            z.extractall(extract_dir)
        print(f"[OK] {name} downloaded and extracted to {extract_dir}")
    except Exception as e:
        print(f"[FAILED] {name}: {e}")

for root, dirs, files in os.walk(BASE):
    for f in files:
        print(os.path.join(root, f))

[OK] dropout downloaded and extracted to /content/EduPredict/data/raw/dropout
[OK] performance downloaded and extracted to /content/EduPredict/data/raw/performance
/content/EduPredict/data/raw/dropout.zip
/content/EduPredict/data/raw/performance.zip
/content/EduPredict/data/raw/performance/student.zip
/content/EduPredict/data/raw/performance/.student.zip_old
/content/EduPredict/data/raw/dropout/data.csv


In [2]:
import os, zipfile
import pandas as pd
from google.colab import files

pd.set_option("display.max_columns", None)
BASE = "/content/EduPredict/data/raw"
os.makedirs(BASE, exist_ok=True)

uploaded = files.upload()   # choose both zip files

for fname in uploaded:
    with zipfile.ZipFile(fname) as z:
        z.extractall(BASE)

# the performance zip contains a nested student.zip
nested = os.path.join(BASE, "student.zip")
if os.path.exists(nested):
    with zipfile.ZipFile(nested) as z:
        z.extractall(BASE)

print(sorted(os.listdir(BASE)))

['dropout', 'dropout.zip', 'performance', 'performance.zip']


In [3]:
for root, dirs, files in os.walk(BASE):
    for f in files:
        if f.endswith(".zip") and root != BASE:
            inner = os.path.join(root, f)
            with zipfile.ZipFile(inner) as z:
                z.extractall(root)
            print("Extracted nested zip:", inner)

csv_files = []
for root, dirs, files in os.walk(BASE):
    for f in files:
        if f.endswith(".csv"):
            csv_files.append(os.path.join(root, f))

print("\nCSV files found:")
for c in csv_files:
    print(" ", c)

Extracted nested zip: /content/EduPredict/data/raw/performance/student.zip

CSV files found:
  /content/EduPredict/data/raw/performance/student-mat.csv
  /content/EduPredict/data/raw/performance/student-por.csv
  /content/EduPredict/data/raw/dropout/data.csv


In [4]:
def find_csv(filename_part):
    for c in csv_files:
        if filename_part in os.path.basename(c).lower():
            return c
    raise FileNotFoundError(f"No CSV containing '{filename_part}' found. Check the file list above.")

df_dropout = pd.read_csv(find_csv("data"), sep=";")
df_math = pd.read_csv(find_csv("student-mat"), sep=";")
df_por  = pd.read_csv(find_csv("student-por"), sep=";")

print("Dropout dataset   :", df_dropout.shape)
print("Math dataset      :", df_math.shape)
print("Portuguese dataset:", df_por.shape)

Dropout dataset   : (4424, 37)
Math dataset      : (395, 33)
Portuguese dataset: (649, 33)


In [5]:
print("=== DROPOUT DATASET ===")
print("Shape:", df_dropout.shape)
display(df_dropout.head())

print("\nColumn names with hidden whitespace:")
print([repr(c) for c in df_dropout.columns if c != c.strip()])

print("\nData types:")
print(df_dropout.dtypes.value_counts())

print("\nMissing values total:", df_dropout.isnull().sum().sum())
print("Duplicate rows:", df_dropout.duplicated().sum())

print("\nTarget counts:")
print(df_dropout["Target"].value_counts())
print("\nTarget percentages:")
print((df_dropout["Target"].value_counts(normalize=True) * 100).round(1))

print("\nGender counts (0/1 coded):")
print(df_dropout["Gender"].value_counts())

=== DROPOUT DATASET ===
Shape: (4424, 37)


,Marital status,Application mode,Application order,Course,Daytime/evening attendance\t,Previous qualification,Previous qualification (grade),Nacionality,Mother's qualification,Father's qualification,Mother's occupation,Father's occupation,Admission grade,Displaced,Educational special needs,Debtor,Tuition fees up to date,Gender,Scholarship holder,Age at enrollment,International,Curricular units 1st sem (credited),Curricular units 1st sem (enrolled),Curricular units 1st sem (evaluations),Curricular units 1st sem (approved),Curricular units 1st sem (grade),Curricular units 1st sem (without evaluations),Curricular units 2nd sem (credited),Curricular units 2nd sem (enrolled),Curricular units 2nd sem (evaluations),Curricular units 2nd sem (approved),Curricular units 2nd sem (grade),Curricular units 2nd sem (without evaluations),Unemployment rate,Inflation rate,GDP,Target
0,1,17,5,171,1,1,122.0,1,19,12,5,9,127.3,1,0,0,1,1,0,20,0,0,0,0,0,0.000000,0,0,0,0,0,0.000000,0,10.8,1.4,1.74,Dropout
1,1,15,1,9254,1,1,160.0,1,1,3,3,3,142.5,1,0,0,0,1,0,19,0,0,6,6,6,14.000000,0,0,6,6,6,13.666667,0,13.9,-0.3,0.79,Graduate
2,1,1,5,9070,1,1,122.0,1,37,37,9,9,124.8,1,0,0,0,1,0,19,0,0,6,0,0,0.000000,0,0,6,0,0,0.000000,0,10.8,1.4,1.74,Dropout
3,1,17,2,9773,1,1,122.0,1,38,37,5,3,119.6,1,0,0,1,0,0,20,0,0,6,8,6,13.428571,0,0,6,10,5,12.400000,0,9.4,-0.8,-3.12,Graduate
4,2,39,1,8014,0,1,100.0,1,37,38,9,9,141.5,0,0,0,1,0,0,45,0,0,6,9,5,12.333333,0,0,6,6,6,13.000000,0,13.9,-0.3,0.79,Graduate



Column names with hidden whitespace:
["'Daytime/evening attendance\\t'"]

Data types:
int64      29
float64     7
object      1
Name: count, dtype: int64

Missing values total: 0
Duplicate rows: 0

Target counts:
Target
Graduate    2209
Dropout     1421
Enrolled     794
Name: count, dtype: int64

Target percentages:
Target
Graduate    49.9
Dropout     32.1
Enrolled    17.9
Name: proportion, dtype: float64

Gender counts (0/1 coded):
Gender
0    2868
1    1556
Name: count, dtype: int64


In [6]:
for name, d in [("MATH", df_math), ("PORTUGUESE", df_por)]:
    print(f"\n=== {name} ===")
    display(d.head())
    print("Shape:", d.shape)
    print("Missing:", d.isnull().sum().sum(), "| Duplicates:", d.duplicated().sum())
    print("Text columns:", list(d.select_dtypes(exclude="number").columns))
    print(d[["G1", "G2", "G3"]].describe().round(2))
    print("Students with G3 == 0:", (d["G3"] == 0).sum())


=== MATH ===


,school,sex,age,address,famsize,Pstatus,Medu,Fedu,Mjob,Fjob,reason,guardian,traveltime,studytime,failures,schoolsup,famsup,paid,activities,nursery,higher,internet,romantic,famrel,freetime,goout,Dalc,Walc,health,absences,G1,G2,G3
0,GP,F,18,U,GT3,A,4,4,at_home,teacher,course,mother,2,2,0,yes,no,no,no,yes,yes,no,no,4,3,4,1,1,3,6,5,6,6
1,GP,F,17,U,GT3,T,1,1,at_home,other,course,father,1,2,0,no,yes,no,no,no,yes,yes,no,5,3,3,1,1,3,4,5,5,6
2,GP,F,15,U,LE3,T,1,1,at_home,other,other,mother,1,2,3,yes,no,yes,no,yes,yes,yes,no,4,3,2,2,3,3,10,7,8,10
3,GP,F,15,U,GT3,T,4,2,health,services,home,mother,1,3,0,no,yes,yes,yes,yes,yes,yes,yes,3,2,2,1,1,5,2,15,14,15
4,GP,F,16,U,GT3,T,3,3,other,other,home,father,1,2,0,no,yes,yes,no,yes,yes,no,no,4,3,2,1,2,5,4,6,10,10


Shape: (395, 33)
Missing: 0 | Duplicates: 0
Text columns: ['school', 'sex', 'address', 'famsize', 'Pstatus', 'Mjob', 'Fjob', 'reason', 'guardian', 'schoolsup', 'famsup', 'paid', 'activities', 'nursery', 'higher', 'internet', 'romantic']
           G1      G2      G3
count  395.00  395.00  395.00
mean    10.91   10.71   10.42
std      3.32    3.76    4.58
min      3.00    0.00    0.00
25%      8.00    9.00    8.00
50%     11.00   11.00   11.00
75%     13.00   13.00   14.00
max     19.00   19.00   20.00
Students with G3 == 0: 38

=== PORTUGUESE ===


,school,sex,age,address,famsize,Pstatus,Medu,Fedu,Mjob,Fjob,reason,guardian,traveltime,studytime,failures,schoolsup,famsup,paid,activities,nursery,higher,internet,romantic,famrel,freetime,goout,Dalc,Walc,health,absences,G1,G2,G3
0,GP,F,18,U,GT3,A,4,4,at_home,teacher,course,mother,2,2,0,yes,no,no,no,yes,yes,no,no,4,3,4,1,1,3,4,0,11,11
1,GP,F,17,U,GT3,T,1,1,at_home,other,course,father,1,2,0,no,yes,no,no,no,yes,yes,no,5,3,3,1,1,3,2,9,11,11
2,GP,F,15,U,LE3,T,1,1,at_home,other,other,mother,1,2,0,yes,no,no,no,yes,yes,yes,no,4,3,2,2,3,3,6,12,13,12
3,GP,F,15,U,GT3,T,4,2,health,services,home,mother,1,3,0,no,yes,no,yes,yes,yes,yes,yes,3,2,2,1,1,5,0,14,14,14
4,GP,F,16,U,GT3,T,3,3,other,other,home,father,1,2,0,no,yes,no,no,yes,yes,no,no,4,3,2,1,2,5,0,11,13,13


Shape: (649, 33)
Missing: 0 | Duplicates: 0
Text columns: ['school', 'sex', 'address', 'famsize', 'Pstatus', 'Mjob', 'Fjob', 'reason', 'guardian', 'schoolsup', 'famsup', 'paid', 'activities', 'nursery', 'higher', 'internet', 'romantic']
           G1      G2      G3
count  649.00  649.00  649.00
mean    11.40   11.57   11.91
std      2.75    2.91    3.23
min      0.00    0.00    0.00
25%     10.00   10.00   10.00
50%     11.00   11.00   12.00
75%     13.00   13.00   14.00
max     19.00   19.00   19.00
Students with G3 == 0: 15


In [7]:
print("DROPOUT columns:")
for i, c in enumerate(df_dropout.columns, 1):
    print(i, repr(c))

print("\nPERFORMANCE columns:")
for i, c in enumerate(df_math.columns, 1):
    print(i, c)

DROPOUT columns:
1 'Marital status'
2 'Application mode'
3 'Application order'
4 'Course'
5 'Daytime/evening attendance\t'
6 'Previous qualification'
7 'Previous qualification (grade)'
8 'Nacionality'
9 "Mother's qualification"
10 "Father's qualification"
11 "Mother's occupation"
12 "Father's occupation"
13 'Admission grade'
14 'Displaced'
15 'Educational special needs'
16 'Debtor'
17 'Tuition fees up to date'
18 'Gender'
19 'Scholarship holder'
20 'Age at enrollment'
21 'International'
22 'Curricular units 1st sem (credited)'
23 'Curricular units 1st sem (enrolled)'
24 'Curricular units 1st sem (evaluations)'
25 'Curricular units 1st sem (approved)'
26 'Curricular units 1st sem (grade)'
27 'Curricular units 1st sem (without evaluations)'
28 'Curricular units 2nd sem (credited)'
29 'Curricular units 2nd sem (enrolled)'
30 'Curricular units 2nd sem (evaluations)'
31 'Curricular units 2nd sem (approved)'
32 'Curricular units 2nd sem (grade)'
33 'Curricular units 2nd sem (without evaluati

In [8]:
final_dir = "/content/EduPredict/data/raw"
os.makedirs(final_dir, exist_ok=True)

df_dropout.to_csv(f"{final_dir}/dropout_raw.csv", index=False)
df_math.to_csv(f"{final_dir}/student_math_raw.csv", index=False)
df_por.to_csv(f"{final_dir}/student_por_raw.csv", index=False)

print(sorted(f for f in os.listdir(final_dir) if f.endswith(".csv")))

['dropout_raw.csv', 'student_math_raw.csv', 'student_por_raw.csv']


In [ ]:
import os
import pandas as pd
import numpy as np
from google.colab import files

pd.set_option("display.max_columns", None)

RAW = "/content/EduPredict/data/raw"
PROC = "/content/EduPredict/data/processed"
os.makedirs(RAW, exist_ok=True)
os.makedirs(PROC, exist_ok=True)

uploaded = files.upload()   # select dropout_raw.csv, student_math_raw.csv, student_por_raw.csv
for name, content in uploaded.items():
    with open(f"{RAW}/{name}", "wb") as f:
        f.write(content)

df_dropout = pd.read_csv(f"{RAW}/dropout_raw.csv")
df_math = pd.read_csv(f"{RAW}/student_math_raw.csv")
df_por = pd.read_csv(f"{RAW}/student_por_raw.csv")
print(df_dropout.shape, df_math.shape, df_por.shape)